# 📊 PIPELINE NLP ANALISIS SENTIMEN & FRAMING MEDIA BERITA (RISET FISIP - SCOPUS)
**Studi Komparatif Pemberitaan Media Regional Terkait Isu Papua & Diplomasi Kemelanesiaan**

Notebook ini mengolah dataset artikel berita regional (`news_papua_asean_articles.csv`) yang mencakup media arus utama di **Singapura (CNA, Straits Times)**, **Malaysia (The Star, FMT, NST, Astro AWANI)**, **Kawasan Pasifik (RNZ, ABC News, Asia Pacific Report)**, dan **Mitra Global (Al Jazeera)**.

### 🎯 Kerangka Teoretis & Rumusan Masalah (`MABA SCOPUS.pdf`):
1. **RQ1 (Entman Framing Theory)**: Memetakan bingkai narasi dominan media (*Define Problems, Diagnose Causes, Make Moral Judgement, Treatment Recommendation*) pada isu HAM, Kedaulatan, Diplomasi, Pembangunan, dan Kemelanesiaan.
2. **RQ2 (Attentive Public)**: Menganalisis eksposur wacana institusional yang dikonsumsi kelompok terdidik dan pemerhati kebijakan publik di **Malaysia dan Singapura**.
3. **RQ3 (Diplomasi Publik Eksternal RI)**: Menilai sejauh mana narasi kedaulatan dan pembangunan Indonesia diterima atau dikritisi oleh pers regional.

### 📦 Langkah 1: Instalasi Pustaka (Transformers, PyTorch, WordCloud)

In [ ]:
# Instalasi dependensi NLP & visualisasi
!pip install transformers torch scipy pandas numpy matplotlib seaborn wordcloud tqdm

### 📂 Langkah 2: Unggah Dataset Media Berita
Unggah berkas `news_papua_asean_articles.csv` dari folder `0_data/` jika Anda menjalankannya di Google Colab.

In [ ]:
import os
import pandas as pd

input_csv = "news_papua_asean_articles.csv"

# Cek jika file belum ada di Colab, buka dialog upload
if not os.path.exists(input_csv):
    print("Silakan unggah berkas 'news_papua_asean_articles.csv':")
    from google.colab import files
    uploaded = files.upload()
    input_csv = list(uploaded.keys())[0]

df = pd.read_csv(input_csv)
print(f"[✓] Berhasil memuat {len(df)} artikel berita!")
display(df[["media_source", "country_bloc", "created_at", "full_text"]].head(3))

### 🧠 Langkah 3: Inisialisasi Model Transformers & Konfigurasi Entman Framing

In [ ]:
import re
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan Perangkat: {device.upper()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("[✓] Pretrained RoBERTa berhasil dimuat!")

# Kamus Framing Entman untuk Wacana Papua & Kawasan
NARRATIVE_CATEGORIES = {
    "HAM & Pelanggaran Kemanusiaan": [
        "human rights", "un commissioner", "rights abuse", "abuses", "genocide", 
        "violations", "investigation", "torture", "civilians", "hak asasi", 
        "pbb", "killings", "atrocities", "massacre", "violence", "racism", 
        "rasis", "brutal", "unrest", "clashes", "displaced", "refugees", "komnas ham"
    ],
    "Identitas Melanesia & Solidaritas Pasifik": [
        "melanesia", "melanesian", "msg", "melanesian spearhead group", "pacific forum", 
        "vanuatu", "solomon islands", "fiji", "kanaky", "solidarity", "brotherhood", 
        "rumpun", "wantok", "black people", "indigenous", "ras", "etnis", "ancestors", 
        "oceania", "cultural identity"
    ],
    "Kedaulatan & Penentuan Nasib Sendiri": [
        "sovereignty", "referendum", "independence", "otsus", "freedom", 
        "decolonisation", "ulmwp", "nkri", "kedaulatan", "otonomi khusus", 
        "merdeka", "pepera", "act of free choice", "self-determination", "separatist", 
        "separatis", "treason", "makar", "balkanization", "territorial integrity"
    ],
    "Pembangunan & Stabilitas Keamanan": [
        "development", "infrastructure", "conflict", "tni", "opm", "tpnpb", 
        "military", "armed group", "hostage", "pilot", "economy", "pembangunan", 
        "infrastruktur", "jokowi", "prabowo", "tambang", "freeport", "nickel", 
        "raja ampat", "carbon capture", "security forces", "police"
    ],
    "Diplomasi & Hubungan Internasional": [
        "diplomacy", "bilateral", "foreign minister", "ambassador", "foreign policy", 
        "united nations", "asean", "treaty", "negotiation", "dialogue", "ties", 
        "international community", "canberra", "wellington", "australia", "new zealand", 
        "pacific islands", "summit", "kemlu"
    ]
}

def clean_news_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'^[A-Z\s]{2,15}:\s*', '', text)
    text = re.sub(r'[^\w\s.,!?-]', ' ', text)
    return " ".join(text.split())

def extract_lead_text(full_text, max_words=250):
    cleaned = clean_news_text(full_text)
    words = cleaned.split()
    return " ".join(words[:max_words])

def adjust_contextual_sentiment_news(text, base_label):
    text_lower = text.lower()
    crit = ["human rights violation", "rights abuses", "genocide", "torture", "illegal occupation", "colonial", "oppression", "internet blackout", "crushed protest", "treason charge", "killed civilians", "destruction of forest", "nickel mining threat", "marginalized", "apartheid", "atrocities"]
    supp = ["carbon storage", "infrastructure project", "economic development", "special autonomy", "sovereignty recognized", "rescued pilot", "freed hostage", "lawful territory", "aid to pacific", "peaceful dialogue", "investing in papua", "recovers body", "firm action against terror"]
    crit_c = sum(1 for kw in crit if kw in text_lower)
    supp_c = sum(1 for kw in supp if kw in text_lower)
    if crit_c > 0 and crit_c >= supp_c:
        return "Negatif (Kritis terhadap Posisi/Diplomasi RI)"
    elif supp_c > 0 and supp_c > crit_c:
        return "Positif (Mendukung Pembangunan/Kedaulatan RI)"
    if base_label == "negative": return "Negatif (Kritis terhadap Posisi/Diplomasi RI)"
    elif base_label == "positive": return "Positif (Mendukung Pembangunan/Kedaulatan RI)"
    return "Netral (Liputan Faktual/Berimbang)"

def classify_entman_framing(text):
    text_lower = text.lower()
    scores = {}
    for cat, kws in NARRATIVE_CATEGORIES.items():
        scores[cat] = sum(1 for kw in kws if re.search(r'\b' + re.escape(kw) + r'\b', text_lower))
    sorted_s = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    if sorted_s[0][1] > 0: return sorted_s[0][0]
    return "Diplomasi & Hubungan Internasional"

### ⚡ Langkah 4: Eksekusi Analisis Sentimen & Entman Framing

In [ ]:
from tqdm import tqdm

labels_map = {0: "negative", 1: "neutral", 2: "positive"}
processed_records = []

print("Memproses analisis NLP pada artikel media berita...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    full_text = str(row.get("full_text", ""))
    lead_text = extract_lead_text(full_text, max_words=250)
    
    base_sentiment = "neutral"
    confidence_score = 0.50
    
    if len(lead_text) > 20:
        try:
            encoded = tokenizer(lead_text, return_tensors='pt', truncation=True, max_length=512).to(device)
            with torch.no_grad():
                output = model(**encoded)
                scores = softmax(output.logits[0].cpu().numpy())
                pred_idx = np.argmax(scores)
                base_sentiment = labels_map[pred_idx]
                confidence_score = float(scores[pred_idx])
        except Exception:
            pass
            
    final_sentiment = adjust_contextual_sentiment_news(full_text, base_sentiment)
    narrative_frame = classify_entman_framing(full_text)
    
    rec = row.to_dict()
    rec["clean_lead"] = lead_text
    rec["sentiment_raw"] = base_sentiment
    rec["sentiment"] = final_sentiment
    rec["sentiment_score"] = round(confidence_score, 4)
    rec["dominant_narrative"] = narrative_frame
    processed_records.append(rec)

res_df = pd.DataFrame(processed_records)
output_csv_analyzed = "news_papua_sentiment_analyzed.csv"
res_df.to_csv(output_csv_analyzed, index=False, encoding='utf-8')
print(f"\n[✓] Selesai! Hasil disimpan ke '{output_csv_analyzed}'.")

### 📈 Langkah 5: Pembuatan Visualisasi Publikasi Ilmiah (300 DPI)
Grafik dihasilkan dengan resolusi tinggi siap pakai untuk artikel jurnal Scopus.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

palette_sentimen = {
    "Negatif (Kritis terhadap Posisi/Diplomasi RI)": "#e74c3c",
    "Positif (Mendukung Pembangunan/Kedaulatan RI)": "#2ecc71",
    "Netral (Liputan Faktual/Berimbang)": "#95a5a6"
}

# 1. PIE CHART SENTIMEN
plt.figure(figsize=(8, 8), dpi=300)
sentiment_counts = res_df["sentiment"].value_counts()
colors = [palette_sentimen.get(label, "#34495e") for label in sentiment_counts.index]
plt.pie(
    sentiment_counts, 
    labels=sentiment_counts.index, 
    autopct='%1.1f%%', 
    startangle=140, 
    colors=colors,
    textprops={'fontsize': 11, 'weight': 'bold'},
    explode=[0.04] * len(sentiment_counts)
)
plt.title("Distribusi Sentimen Media Berita Online Regional\n(Isu Papua & Identitas Kemelanesiaan)", fontsize=14, weight='bold', pad=20)
plt.savefig("news_sentiment_distribution_pie.png", bbox_inches='tight')
plt.show()

# 2. BAR CHART FRAMING NARASI
plt.figure(figsize=(10, 6), dpi=300)
narrative_counts = res_df["dominant_narrative"].value_counts()
ax = sns.barplot(x=narrative_counts.values, y=narrative_counts.index, palette="mako")
plt.title("Distribusi Narasi Dominan Media Berita (Entman Framing)", fontsize=14, weight='bold', pad=15)
plt.xlabel("Jumlah Artikel Berita", fontsize=12)
plt.ylabel("Kategori Framing Narasi", fontsize=12)
for i, v in enumerate(narrative_counts.values):
    ax.text(v + 1, i, str(v), color='black', va='center', weight='bold')
plt.savefig("news_narrative_distribution_bar.png", bbox_inches='tight')
plt.show()

# 3. STACKED BAR CHART SENTIMEN PER BLOK
plt.figure(figsize=(10, 6), dpi=300)
cross_bloc = pd.crosstab(res_df["country_bloc"], res_df["sentiment"], normalize='index') * 100
cols_order = [col for col in palette_sentimen.keys() if col in cross_bloc.columns]
cross_bloc = cross_bloc[cols_order]
cross_bloc.plot(kind='barh', stacked=True, color=[palette_sentimen[c] for c in cols_order], figsize=(10, 6))
plt.title("Proporsi Sentimen Pemberitaan per Blok Geopolitik", fontsize=14, weight='bold', pad=15)
plt.xlabel("Persentase (%)", fontsize=12)
plt.ylabel("Blok Geopolitik", fontsize=12)
plt.legend(title="Sentimen", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig("news_sentiment_by_bloc_bar.png", bbox_inches='tight')
plt.show()

# 4. BAR CHART SENTIMEN PER MEDIA
plt.figure(figsize=(12, 7), dpi=300)
media_sent_df = res_df.groupby(["media_source", "sentiment"]).size().unstack(fill_value=0)
media_cols = [col for col in palette_sentimen.keys() if col in media_sent_df.columns]
media_sent_df = media_sent_df[media_cols]
media_sent_df.plot(kind='barh', stacked=True, color=[palette_sentimen[c] for c in media_cols], figsize=(12, 7))
plt.title("Distribusi Sentimen per Portal Media Berita Resmi", fontsize=14, weight='bold', pad=15)
plt.xlabel("Jumlah Artikel", fontsize=12)
plt.ylabel("Kanal Media Berita", fontsize=12)
plt.legend(title="Sentimen", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.savefig("news_sentiment_by_media_bar.png", bbox_inches='tight')
plt.show()

# 5. WORDCLOUD
custom_stops = set(STOPWORDS).union({
    "said", "says", "told", "reported", "statement", "news", "reuters", "afp", 
    "cna", "al", "jazeera", "abc", "star", "fmt", "rnz", "pacific", "times", 
    "article", "report", "source", "spokesman", "official", "government", 
    "indonesia", "indonesian", "papua", "west", "one", "two", "three", "also", 
    "will", "would", "could", "may", "can", "now", "year", "years", "month", 
    "people", "time", "first", "last", "new", "day", "days", "jakarta", "region"
})
all_clean_text = " ".join(res_df["clean_lead"].tolist())
wordcloud = WordCloud(width=1200, height=600, background_color="white", stopwords=custom_stops, colormap="inferno", max_words=120).generate(all_clean_text)
plt.figure(figsize=(14, 7), dpi=300)
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("WordCloud Kata Kunci Dominan Media Berita Online Regional", fontsize=16, weight='bold', pad=20)
plt.savefig("news_wordcloud_articles.png", bbox_inches='tight')
plt.show()

### 📝 Langkah 6: Ringkasan Temuan Riset FISIP (Scopus)

In [ ]:
summary_text = f"""===============================================================================
     LAPORAN TEMUAN NLP MEDIA BERITA ONLINE REGIONAL (RISET FISIP)
===============================================================================
Total Artikel Berita Dianalisis : {len(res_df)} Artikel

1. DISTRIBUSI SENTIMEN MEDIA REGIONAL TERHADAP DIPLOMASI RI:
"""
for sent, count in res_df["sentiment"].value_counts().items():
    pct = round((count / len(res_df)) * 100, 2)
    summary_text += f"   - {sent:<45}: {count} artikel ({pct}%)\n"

summary_text += "\n2. PEMETAAN NARASI DOMINAN (ENTMAN FRAMING THEORY):\n"
for narr, count in res_df["dominant_narrative"].value_counts().items():
    pct = round((count / len(res_df)) * 100, 2)
    summary_text += f"   - {narr:<45}: {count} artikel ({pct}%)\n"

summary_text += "\n3. SEBARAN PER BLOK GEOPOLITIK MEDIA:\n"
for bloc, count in res_df["country_bloc"].value_counts().items():
    pct = round((count / len(res_df)) * 100, 2)
    summary_text += f"   - {bloc:<30}: {count} artikel ({pct}%)\n"

summary_text += "\n4. SEBARAN PER KANAL MEDIA BERITA RESMI:\n"
for media, count in res_df["media_source"].value_counts().items():
    pct = round((count / len(res_df)) * 100, 2)
    summary_text += f"   - {media:<30}: {count} artikel ({pct}%)\n"

summary_text += """===============================================================================
KESIMPULAN METODOLOGIS UNTUK PUBLIKASI SCOPUS:
Pemberitaan media institusional regional di Blok ASEAN (Singapura & Malaysia)
menunjukkan porsi liputan faktual/berimbang yang tinggi, sementara media di
kawasan Pasifik & Melanesia (RNZ Pacific, ABC) lebih dominan mengangkat framing
HAM dan hak penentuan nasib sendiri dalam dinamika diplomasi MSG.
===============================================================================
"""
print(summary_text)
with open("ringkasan_analisis_news_fisip.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

### 📥 Langkah 7: Unduh Semua Hasil Analisis & Visualisasi

In [ ]:
from google.colab import files

# Unduh file CSV hasil analisis dan laporan teks
files.download("news_papua_sentiment_analyzed.csv")
files.download("ringkasan_analisis_news_fisip.txt")

# Unduh visualisasi PNG
for img in ["news_sentiment_distribution_pie.png", "news_narrative_distribution_bar.png", "news_sentiment_by_bloc_bar.png", "news_sentiment_by_media_bar.png", "news_wordcloud_articles.png"]:
    files.download(img)